In [141]:
# Connect to postgres
import psycopg2
import pandas as pd
import numpy as np
import warnings
import os
from dotenv import load_dotenv
warnings.filterwarnings("ignore")
from datetime import date

load_dotenv()

n_conn = psycopg2.connect(
    database=os.getenv("PG_DATABASE"),
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=os.getenv("PG_PORT", "5432")
)

n_cursor = n_conn.cursor()
print("done")


done


In [2]:
# connect to usu vpn first
import pymssql
import os
from dotenv import load_dotenv

load_dotenv()

o_conn = pymssql.connect(
    server=os.getenv("MSSQL_SERVER"),
    user=os.getenv("MSSQL_USER"),
    password=os.getenv("MSSQL_PASSWORD"),
    database=os.getenv("MSSQL_DATABASE")
)

o_cursor = o_conn.cursor()
print("done")


done


In [ ]:
# ran this in Beekeeper because it wouldn't run in here
'''CREATE DATABASE iwdm'''

In [3]:
n_cursor.execute('''
    CREATE TABLE IF NOT EXISTS exception (
    exception_id SERIAL PRIMARY KEY,
    row_data TEXT NOT NULL,
    filename VARCHAR(50)
);
''')
n_conn.commit()
print("done")

done


In [10]:
#Customer Table DDL:
n_cursor.execute('''
    CREATE TABLE IF NOT EXISTS Customer (
    customer_id SERIAL PRIMARY KEY,
    first_name VARCHAR(25) NOT NULL,
    last_name VARCHAR(25),
    age SMALLINT,
    customer_type VARCHAR(10),
    customer_since SMALLINT,
    customer_income INT,
    household_size SMALLINT,
    mode_color VARCHAR(6)
    );
''')
n_conn.commit()

#Create a dataframe:
df = pd.read_sql("""SELECT * FROM Customer_table;""", o_conn)
#Clean the data:

#Split customer_name column into first_name and last_name
df[['first_name', 'last_name']] = df['customer_name'].str.split(n=1, expand=True)

# Ingest into the database:
for i in df.index:
    first_name = df.loc[i]['first_name']
    last_name = df.loc[i]['last_name']
    age = int(df.loc[i]['customer_age'])
    customer_type = df.loc[i]['customer_type']
    customer_since = int(df.loc[i]['customer_since'])
    customer_income = int(df.loc[i]['customer_income'])
    household_size = int(df.loc[i]['household_size'])
    mode_color = df.loc[i]['mode_color']

    n_cursor.execute('''
        INSERT INTO customer (first_name, last_name, age, customer_type, customer_since, customer_income, household_size, mode_color)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
    ''', (first_name, last_name, age, customer_type, customer_since, customer_income, household_size, mode_color))

n_conn.commit()
print("Ingestion for the customer table is complete")

Ingestion for the customer table is complete


In [11]:
#Teams DDL and DML:
#Teams Table DDL:
n_cursor.execute('''
    CREATE TABLE IF NOT EXISTS teams (
    team_pk SERIAL PRIMARY KEY,
    team_name VARCHAR(30) NOT NULL,
    team_name_short VARCHAR(20) NOT NULL,
    team_id VARCHAR(3) NOT NULL,
    team_id_pfr CHAR(3) NOT NULL,
    team_conference CHAR(3),
    team_division VARCHAR(15),
    team_conference_pre2002 CHAR(3),
    team_division_pre2002 VARCHAR(15)
    );
''')
n_conn.commit()

#Create the dataframe
df = pd.read_csv('nfl_teams.csv')

#Clean the data
df = df.where(pd.notnull(df), None)

#Ingest into the database:
for i in df.index:
    team_name = df.loc[i]['team_name']
    team_name_short = df.loc[i]['team_name_short']
    team_id = df.loc[i]['team_id']
    team_id_pfr = df.loc[i]['team_id_pfr']
    team_conference = df.loc[i]['team_conference']
    team_division = df.loc[i]['team_division']
    team_conference_pre2002 = df.loc[i]['team_conference_pre2002']
    team_division_pre2002 = df.loc[i]['team_division_pre2002']
    #Insert into teams table here!!!
    n_cursor.execute('''
        INSERT INTO teams (team_name, team_name_short, team_id, team_id_pfr, team_conference, team_division, team_conference_pre2002, team_division_pre2002)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
    ''', (team_name, team_name_short, team_id, team_id_pfr, team_conference, team_division, team_conference_pre2002, team_division_pre2002))
    print("Record Created For:", team_name)
n_conn.commit()

Record Created For: Arizona Cardinals
Record Created For: Atlanta Falcons
Record Created For: Baltimore Colts
Record Created For: Baltimore Ravens
Record Created For: Boston Patriots
Record Created For: Buffalo Bills
Record Created For: Carolina Panthers
Record Created For: Chicago Bears
Record Created For: Cincinnati Bengals
Record Created For: Cleveland Browns
Record Created For: Dallas Cowboys
Record Created For: Denver Broncos
Record Created For: Detroit Lions
Record Created For: Green Bay Packers
Record Created For: Houston Oilers
Record Created For: Houston Texans
Record Created For: Indianapolis Colts
Record Created For: Jacksonville Jaguars
Record Created For: Kansas City Chiefs
Record Created For: Las Vegas Raiders
Record Created For: Los Angeles Chargers
Record Created For: Los Angeles Raiders
Record Created For: Los Angeles Rams
Record Created For: Miami Dolphins
Record Created For: Minnesota Vikings
Record Created For: New England Patriots
Record Created For: New Orleans Sa

In [6]:
#Stadium DDL and DML:
#Stadium Table DDL:
n_cursor.execute('''
    CREATE TABLE IF NOT EXISTS stadium (
    stadium_id SERIAL PRIMARY KEY,
    stadium_name VARCHAR(50) NOT NULL,
    stadium_location VARCHAR(40),
    stadium_open SMALLINT,
    stadium_close SMALLINT,
    stadium_type VARCHAR(12),
    stadium_address VARCHAR(120),
    stadium_capacity INT,
    stadium_surface VARCHAR(25),
    stadium_weather_station_code VARCHAR(25),
    stadium_weather_type VARCHAR(10),
    station CHAR(11),
    station_name VARCHAR(55),
    latitude NUMERIC(8,5),
    longitude NUMERIC(8,5),
    elevation NUMERIC(5,1)
    );
''')
n_conn.commit()
#Create a dataframe:
df = pd.read_csv('nfl_stadiums.csv')
print(df.columns)
#Clean the data:
df = df.where(pd.notnull(df), None)
df['stadium_open'] = df['stadium_open'].astype(object)
df['stadium_open'] = df['stadium_open'].where(df['stadium_open'].notna(), None)

df['stadium_close'] = df['stadium_close'].astype(object)
df['stadium_close'] = df['stadium_close'].where(df['stadium_close'].notna(), None)

df['LATITUDE'] = df['LATITUDE'].astype(object)
df['LATITUDE'] = df['LATITUDE'].where(df['LATITUDE'].notna(), None)

df['LONGITUDE'] = df['LONGITUDE'].astype(object)
df['LONGITUDE'] = df['LONGITUDE'].where(df['LONGITUDE'].notna(), None)

df['ELEVATION'] = df['ELEVATION'].astype(object)
df['ELEVATION'] = df['ELEVATION'].where(df['ELEVATION'].notna(), None)

#Stadium_address cleaning:

#Stadium_weather_station_code cleaning:
#1. 4 digit zipcodes (in the loop)
#2. words in the column
#3. 

#stadium_capacity cleaning: Get rid of commas in the column
df['stadium_capacity'] = df['stadium_capacity'].str.replace(',', '')

#Mexico city character
df.loc[24, 'stadium_address'] = 'Calzada de Tlalpan 3465, Santa Ursula Coapa, Coyoacan, 04650 Ciudad de Mexico'

# Ingest into the database:
for i in df.index:
    stadium_name = df.loc[i]['stadium_name']
    stadium_location = df.loc[i]['stadium_location']
    try:
        stadium_open = int(df.loc[i]['stadium_open'])
    except:
        stadium_open = df.loc[i]['stadium_open']
    try:
        stadium_close = int(df.loc[i]['stadium_close'])
    except:
        stadium_close = df.loc[i]['stadium_close']
    stadium_type = df.loc[i]['stadium_type']
    stadium_address = df.loc[i]['stadium_address']
    try:
        stadium_capacity = int(df.loc[i]['stadium_capacity'])
    except:
        stadium_capacity = df.loc[i]['stadium_capacity']
    stadium_surface = df.loc[i]['stadium_surface']
    stadium_weather_station_code = df.loc[i]['stadium_weather_station_code']
    stadium_weather_type = df.loc[i]['stadium_weather_type']
    station = df.loc[i]['STATION']
    station_name = df.loc[i]['NAME']
    try:
        latitude = float(df.loc[i]['LATITUDE'])
    except:
        latitude = df.loc[i]['LATITUDE']
    try:
        longitude = float(df.loc[i]['LONGITUDE'])
    except:
        longitude = df.loc[i]['LONGITUDE']
    try:
        elevation = float(df.loc[i]['ELEVATION'])
    except:
        elevation = df.loc[i]['ELEVATION']
    if stadium_weather_type in ("None", "", "nan", "NaN"):
        stadium_weather_type = None

    #More cleaning:
    code = str(stadium_weather_station_code).strip() if stadium_weather_station_code is not None else None
    if code in (None, "", "None", "nan", "NaN"):
        stadium_weather_station_code = None
    elif code.isdigit():
        if len(code) == 4:
            stadium_weather_station_code = code.zfill(5)
        elif len(code) == 5:
            stadium_weather_station_code = code
        else:
            stadium_weather_station_code = None
    else:
        # Catches non-numeric codes like "Mexico City, MX", "Heathrow, UK"
        row_data = f"{stadium_name}|{stadium_location}|{stadium_open}|{stadium_close}|{stadium_type}|{stadium_address}|{stadium_capacity}|{stadium_surface}|{stadium_weather_station_code}|{stadium_weather_type}|{station}|{station_name}|{latitude}|{longitude}|{elevation}"
        n_cursor.execute('''
            INSERT INTO exception (row_data, filename)
            VALUES (%s, %s)
        ''', (row_data, 'nfl_stadiums.csv'))
        continue 
    
    n_cursor.execute('''
        INSERT INTO stadium (stadium_name, stadium_location, stadium_open, stadium_close, stadium_type, stadium_address, stadium_capacity, stadium_surface, stadium_weather_station_code, stadium_weather_type, station, station_name, latitude, longitude, elevation)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    ''', (stadium_name, stadium_location, stadium_open, stadium_close, stadium_type, stadium_address, stadium_capacity, stadium_surface, stadium_weather_station_code, stadium_weather_type, station, station_name, latitude, longitude, elevation))
    print(f"Record Created For: {stadium_name}")
n_conn.commit()

Index(['stadium_name', 'stadium_location', 'stadium_open', 'stadium_close',
       'stadium_type', 'stadium_address', 'stadium_weather_station_code',
       'stadium_weather_type', 'stadium_capacity', 'stadium_surface',
       'STATION', 'NAME', 'LATITUDE', 'LONGITUDE', 'ELEVATION'],
      dtype='object')
Record Created For: Acrisure Stadium
Record Created For: Alamo Dome
Record Created For: Allegiant Stadium
Record Created For: Allianz Arena
Record Created For: Alltel Stadium
Record Created For: Alumni Stadium
Record Created For: Anaheim Stadium
Record Created For: Arrowhead Stadium
Record Created For: AT&T Stadium
Record Created For: Atlanta-Fulton County Stadium
Record Created For: Balboa Stadium
Record Created For: Bank of America Stadium
Record Created For: Bills Stadium
Record Created For: Busch Memorial Stadium
Record Created For: Caesars Superdome
Record Created For: Candlestick Park
Record Created For: CenturyLink Field
Record Created For: Cinergy Field
Record Created For: Cle

In [7]:
#Schedule DDL and DML:
#schedule Table DDL:
n_cursor.execute('''
    CREATE TABLE IF NOT EXISTS schedule (
    game_id SERIAL PRIMARY KEY,
    game_code VARCHAR(30) NOT NULL,
    game_date DATE,
    schedule_season SMALLINT,
    schedule_week CHAR(2),
    schedule_playoff BOOLEAN,
    home_team_id VARCHAR(3) NOT NULL,
    score_home SMALLINT,
    away_team_id VARCHAR(3) NOT NULL,
    score_away SMALLINT,
    team_favorite_id VARCHAR(4) NOT NULL,
    spread_favorite NUMERIC(3,1),
    over_under_line NUMERIC(3,1),
    stadium_id INT,
    stadium_neutral BOOLEAN,
    weather_temp SMALLINT,
    weather_wind_mph SMALLINT,
    weather_humidity SMALLINT,
    weather_detail VARCHAR(25),
    winner_line CHAR(4),
    winner_ou VARCHAR(5)
    );
''')

n_conn.commit()
#Create a dataframe:
df_pre = pd.read_csv('spread_scores-2.csv')
df = df_pre[df_pre['schedule_season'] >= 2015]

#Clean the data where possible:
df = df.where(pd.notnull(df), None)
#Convert these floats to objects to turn NaN to None (weather_humidity...)
df['weather_humidity'] = df['weather_humidity'].astype(object)
df['weather_humidity'] = df['weather_humidity'].where(df['weather_humidity'].notna(), None)

df['weather_temperature'] = df['weather_temperature'].astype(object)
df['weather_temperature'] = df['weather_temperature'].where(df['weather_temperature'].notna(), None)

df['weather_wind_mph'] = df['weather_wind_mph'].astype(object)
df['weather_wind_mph'] = df['weather_wind_mph'].where(df['weather_wind_mph'].notna(), None)

#Calculate winner_ou (using np.select)
ou_conditions = [(df['score_home'] + df['score_away'] > df['over_under_line']),
                 (df['score_home'] + df['score_away'] < df['over_under_line']),
                 (df['score_home'] + df['score_away'] == df['over_under_line'])]
ou_values = ['over', 'under', 'push']

df['winner_ou'] = np.select(ou_conditions, ou_values, default=None)

# winner line
def compute_winner_line(row, home_team_id, away_team_id):
    # Check if favorite matches home or away team
    fav_is_home = row['team_favorite_id'] == home_team_id
    
    margin = row['score_home'] - row['score_away']
    
    if fav_is_home:
        adjusted_margin = margin + row['spread_favorite']
    else:  # favorite is away
        adjusted_margin = margin - row['spread_favorite']
    
    # Determine who covered the spread
    if adjusted_margin > 0:
        return 'home'
    elif adjusted_margin < 0:
        return 'away'
    else:
        return 'push'
        
# update schedule week variables for gamecode
conditions = [(df['schedule_week'] == '1'),
              (df['schedule_week'] == '2'),
              (df['schedule_week'] == '3'),
              (df['schedule_week'] == '4'),
              (df['schedule_week'] == '5'),
              (df['schedule_week'] == '6'),
              (df['schedule_week'] == '7'),
              (df['schedule_week'] == '8'),
              (df['schedule_week'] == '9'),
              (df['schedule_week'] == '10'),
              (df['schedule_week'] == '11'),
              (df['schedule_week'] == '12'),
              (df['schedule_week'] == '13'),
              (df['schedule_week'] == '14'),
              (df['schedule_week'] == '15'),
              (df['schedule_week'] == '16'),
              (df['schedule_week'] == '17'),
              (df['schedule_week'] == '18'),
              (df['schedule_week'] == 'Wildcard'),
              (df['schedule_week'] == 'Division'),
              (df['schedule_week'] == 'Conference'),
              (df['schedule_week'] == 'Superbowl')]

values = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22']

df['schedule_week'] = np.select(conditions, values)

#Ingest into the database:
for i in df.index:
    game_date = df.loc[i]['schedule_date']
    schedule_season = int(df.loc[i]['schedule_season'])
    schedule_week = df.loc[i]['schedule_week']
    schedule_playoff = bool(df.loc[i]['schedule_playoff'])
    home_team = df.loc[i]['team_home']
    score_home = int(df.loc[i]['score_home'])
    away_team = df.loc[i]['team_away']
    score_away = int(df.loc[i]['score_away'])
    team_favorite_id = df.loc[i]['team_favorite_id']
    spread_favorite = float(df.loc[i]['spread_favorite'])
    over_under_line = float(df.loc[i]['over_under_line'])
    stadium = df.loc[i]['stadium']
    stadium_neutral = bool(df.loc[i]['stadium_neutral'])
    try:
        weather_temp = int(df.loc[i]['weather_temperature'])
    except:
        weather_temp = df.loc[i]['weather_temperature']
    try:
        weather_wind_mph = int(df.loc[i]['weather_wind_mph'])
    except:
        weather_wind_mph = df.loc[i]['weather_wind_mph']
    try:
        weather_humidity = int(df.loc[i]['weather_humidity'])
    except:
        weather_humidity = df.loc[i]['weather_humidity']
    weather_detail = df.loc[i]['weather_detail']
    winner_ou = df.loc[i]['winner_ou']

    #Get the value for home_team_id using home_team
    n_cursor.execute('''
        SELECT team_id
        FROM teams
        WHERE team_name = %s
    ''', (home_team,))
    home_team_id = n_cursor.fetchone()[0]
    home_team_id = home_team_id.upper()
    
    #Get the value for away_team_id using away_team
    n_cursor.execute('''
        SELECT team_id
        FROM teams
        WHERE team_name = %s
    ''', (away_team,))
    away_team_id = n_cursor.fetchone()[0]
    away_team_id = away_team_id.upper()

    # call winner line after getting Id's 
    winner_line = compute_winner_line(df.loc[i], home_team_id, away_team_id)
    
    #Get the value of game_code
    game_code = f"{schedule_season}{schedule_week}-{home_team_id}-{away_team_id}"
    
    #Get the value for stadium_id using stadium
    try:
        n_cursor.execute('''
        SELECT stadium_id
        FROM stadium
        WHERE stadium_name = %s
        ''', (stadium,))
        stadium_id = int(n_cursor.fetchone()[0])
    except:
        stadium_id = None
        # row_data = f"{game_code}|{game_date}|{schedule_season}|{schedule_week}|{schedule_playoff}|{home_team_id}|{score_home}|{away_team_id}|{score_away}|{team_favorite_id}|{spread_favorite}|{over_under_line}|{stadium_id}|{stadium_neutral}|{weather_temp}|{weather_wind_mph}|{weather_humidity}|{weather_detail}|{winner_line}|{winner_ou}"
        # n_cursor.execute('''
        #     INSERT INTO exception (row_data, filename)
        #     VALUES (%s, 'spread_scores-2.csv')
        # ''', (row_data,))
    #Insert into the table
    try:
        n_cursor.execute('''
            INSERT INTO schedule (game_code, game_date, schedule_season, schedule_week, schedule_playoff, home_team_id, score_home, away_team_id, score_away, team_favorite_id, spread_favorite, over_under_line, stadium_id, stadium_neutral, weather_temp, weather_wind_mph, weather_humidity, weather_detail, winner_line, winner_ou)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s,%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ''', (game_code, game_date, schedule_season, schedule_week, schedule_playoff, home_team_id, score_home, away_team_id, score_away, team_favorite_id, spread_favorite, over_under_line, stadium_id, stadium_neutral, weather_temp, weather_wind_mph, weather_humidity, weather_detail, winner_line, winner_ou))
        n_conn.commit()
    except:
        print(game_code, game_date, schedule_season, schedule_week, schedule_playoff, home_team_id, score_home, away_team_id, score_away, team_favorite_id, spread_favorite, over_under_line, stadium_id, stadium_neutral, weather_temp, weather_wind_mph, weather_humidity, weather_detail, winner_line, winner_ou)
print("Ingestion Completed")    

Ingestion Completed


In [44]:
# Connect to postgres
import psycopg2
import pandas as pd
import numpy as np
import warnings
import os
from dotenv import load_dotenv
warnings.filterwarnings("ignore")
from datetime import date

load_dotenv()

n_conn = psycopg2.connect(
    database=os.getenv("PG_DATABASE"),
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=os.getenv("PG_PORT", "5432")
)

n_cursor = n_conn.cursor()
print("done")


Skipped 100 existing records so far...
Skipped 200 existing records so far...
Skipped 300 existing records so far...
Skipped 400 existing records so far...
Skipped 500 existing records so far...
Skipped 600 existing records so far...
Skipped 700 existing records so far...
Skipped 800 existing records so far...
Skipped 900 existing records so far...
Skipped 1000 existing records so far...
Skipped 1100 existing records so far...
Skipped 1200 existing records so far...
Skipped 1300 existing records so far...
Skipped 1400 existing records so far...
Skipped 1500 existing records so far...
Skipped 1600 existing records so far...
Skipped 1700 existing records so far...
Skipped 1800 existing records so far...
Skipped 1900 existing records so far...
Skipped 2000 existing records so far...
Skipped 2100 existing records so far...
Skipped 2200 existing records so far...
Skipped 2300 existing records so far...
Skipped 2400 existing records so far...
Skipped 2500 existing records so far...
Skipped 2

In [17]:
# TEST QUERIES

In [134]:
# 1
tables = ['betting_log', 'customer', 'exception', 'schedule', 'stadium', 'teams']

for table in tables:
    query = '''SELECT * FROM %s LIMIT 5;''' % (table)
    df = pd.read_sql(query, n_conn)
    print('\n', table)
    print(df)


 betting_log
   bet_id  customer_id      game_code              bet_on  bet_amount result  \
0   36369            1  202301-KC-DET       Detroit Lions        8500    win   
1   36370            8  202301-KC-DET  Kansas City Chiefs       10000   loss   
2   36371           27  202301-KC-DET  Kansas City Chiefs        1100   loss   
3   36372           32  202301-KC-DET       Detroit Lions         500    win   
4   36373           43  202301-KC-DET  Kansas City Chiefs        1000   loss   

   commission  
0       630.0  
1       720.0  
2       108.0  
3        50.0  
4       100.0  

 customer
   customer_id first_name last_name  age customer_type  customer_since  \
0            1   Coraline    Flores   23        online            2022   
1            2    Presley     Ortiz   41        online            2021   
2            3       Reid    Knight   40        online            2023   
3            4   Clarissa  Chandler   43         local            2023   
4            5      Isaac   

In [135]:
# 2

# Query 1
query1 = '''SELECT 
    (SELECT COUNT(*) 
     FROM (SELECT customer_id
           FROM betting_log
           GROUP BY customer_id
           HAVING SUM(commission) > 20000) AS over20k) AS cust_over20k,
    
    (SELECT COUNT(DISTINCT customer_id) 
     FROM betting_log) AS total_customer,
    
    ROUND(
        CAST((SELECT COUNT(*) 
              FROM (SELECT customer_id
                    FROM betting_log
                    GROUP BY customer_id
                    HAVING SUM(commission) > 20000) AS over20k) AS NUMERIC) / 
        CAST((SELECT COUNT(DISTINCT customer_id) 
              FROM betting_log) AS NUMERIC)
    , 2) AS ratio;'''
df = pd.read_sql(query1, n_conn)
print("Query 1: ", '\n', df)
    

# Query 2
query2 = '''SELECT c.first_name, c.last_name, SUM(b.commission) as total_commissions_paid
        FROM customer c 
          JOIN betting_log b on c.customer_id = b.customer_id
        GROUP BY c.first_name, c.last_name
        HAVING SUM(b.commission) > 20000
        ORDER BY total_commissions_paid desc
        limit 20; '''

df = pd.read_sql(query2, n_conn)
print("Query 2: ", '\n', df)

Query 1:  
    cust_over20k  total_customer  ratio
0           236            2000   0.12
Query 2:  
    first_name last_name  total_commissions_paid
0        Gary   McClain                160114.5
1     Matthew     Booth                147968.5
2      Vaughn     Ortiz                126964.0
3      Bailey    Sparks                123294.0
4       Alexa   Mendoza                122644.0
5      Violet    Tanner                110888.0
6       Nadia     Frank                107458.0
7     Harmoni   Burnett                 98722.5
8       Simon     Cohen                 98533.5
9       Lucas    Wagner                 96160.5
10      Clark    Jordan                 93224.5
11      April    Martin                 93212.5
12       Kate     Noble                 89231.0
13       Lara   Hoffman                 89225.0
14    Winston     Short                 86980.0
15     Chanel      Pope                 86525.5
16     Austin    Flores                 86287.5
17     Warren   Barnett           

In [136]:
# 3
query = '''WITH cte AS (
      SELECT b.customer_id, c.first_name, c.last_name, COUNT(b.bet_id) AS bets_placed, CAST(SUM(2* b.bet_amount) AS MONEY) AS amount_won
      FROM customer AS c JOIN betting_log AS b
        ON c.customer_id = b.customer_id
      GROUP BY b.customer_id, c.first_name, c.last_name
      HAVING COUNT(bet_id) >=6
    ),
    cte2 AS (
      SELECT ct.customer_id, ct.first_name, ct.last_name, COUNT(b.bet_id) AS number_of_bets_won, CAST(SUM(2* b.bet_amount) AS MONEY) AS amount_won
      FROM cte AS ct JOIN betting_log AS b
        ON ct.customer_id = b.customer_id
      WHERE b.result = 'win'
      GROUP BY ct.customer_id, ct.first_name, ct.last_name
    ),
    cte3 AS (
      SELECT b.customer_id, COUNT(b.bet_id) AS total_bets_placed
      FROM betting_log AS b
      GROUP BY b.customer_id
    )
    SELECT ct2.first_name, ct2.last_name, ct3.total_bets_placed, ct2.number_of_bets_won, (CAST(ct2.number_of_bets_won AS float)/ct3.total_bets_placed) AS win_percentage, ct2.amount_won
    FROM cte AS ct JOIN cte2 AS ct2
      ON ct.customer_id = ct2.customer_id
        JOIN cte3 as ct3
          ON ct3.customer_id = ct2.customer_id
    ORDER BY win_percentage DESC, ct.amount_won DESC
    LIMIT 10;'''
df = pd.read_sql(query, n_conn)
print(df)

  first_name last_name  total_bets_placed  number_of_bets_won  win_percentage  \
0  Anastasia    Powell                  6                   6        1.000000   
1      Caleb  Reynolds                126                 119        0.944444   
2      Grace      Ford                 11                  10        0.909091   
3   Penelope      Diaz                 11                  10        0.909091   
4      Henry  Anderson                  9                   8        0.888889   
5    Valerie   Wheeler                  8                   7        0.875000   
6  Alexander   Sherman                  7                   6        0.857143   
7       John       Lee                 13                  11        0.846154   
8    Ezekiel  Martinez                 13                  11        0.846154   
9    Sabrina    Wilson                 12                  10        0.833333   

    amount_won  
0    $1,650.00  
1  $187,950.00  
2   $27,100.00  
3    $7,250.00  
4    $7,300.00  
5    $

In [137]:
# 4
query = ('''
    SELECT b.customer_id, c.first_name, c.last_name,
  CAST(SUM(CASE result
    WHEN 'win' THEN b.commission - (b.bet_amount * 2)
    WHEN 'loss' THEN b.commission + b.bet_amount
    ELSE b.commission - b.bet_amount
  END) AS MONEY) AS net_loss
FROM customer AS c JOIN betting_log AS b
  ON c.customer_id = b.customer_id
GROUP BY b.customer_id, c.first_name, c.last_name
ORDER BY net_loss ASC
LIMIT 20;
''')

df = pd.read_sql(query, n_conn, index_col='customer_id')
print(df)

            first_name last_name        net_loss
customer_id                                     
1639            Bailey    Sparks  -$1,184,356.00
45                Kate     Noble  -$1,146,819.00
1834           Adeline    Conner    -$843,753.50
598             Chanel      Pope    -$834,874.50
994               Lara   Hoffman    -$820,725.00
29             Matthew     Booth    -$794,156.50
536               Owen  Garrison    -$793,080.00
1574           Jackson   Fischer    -$770,923.00
373             Lennox      Bean    -$766,456.00
8              Winston     Short    -$764,845.00
145             Vaughn     Ortiz    -$735,086.00
95                Ryan      Lane    -$722,618.00
1410             Logan     Riley    -$696,946.00
1181             Clark    Jordan    -$689,575.50
420              Alexa   Mendoza    -$678,706.00
440              Tiana  Lawrence    -$668,812.50
180                Lia     Drake    -$656,352.00
1726           Delilah      Bush    -$644,548.50
1725            Aust

In [138]:
# 5

query = '''WITH CTE AS (
SELECT game_code, 
  SUM(CASE WHEN result = 'loss' THEN bet_amount ELSE 0 END) AS house_keeps,
  SUM(CASE WHEN result = 'win' THEN (bet_amount*2) ELSE 0 END) AS house_pays
FROM betting_log b
GROUP BY game_code
ORDER BY game_code)  
  SELECT SUBSTRING(c.game_code, 5, 2) as week, 
    COUNT(*) as total_games, 
    COUNT(CASE WHEN c.house_keeps > c.house_pays THEN 1 END) as winner_games,
    COUNT(CASE WHEN c.house_pays > c.house_keeps THEN 1 END) as loser_games,
    ROUND(CAST(COUNT(CASE WHEN c.house_keeps > c.house_pays THEN 1 END) AS NUMERIC) / COUNT (*), 5) as percentage_of_winners,
    ROUND(CAST(COUNT(CASE WHEN c.house_pays > c.house_keeps THEN 1 END) AS NUMERIC) / COUNT (*), 5) as percentage_of_losers
  FROM CTE as c 
  GROUP BY SUBSTRING(c.game_code, 5, 2)
  ORDER BY week;'''

df = pd.read_sql(query, n_conn)
print(df)

   week  total_games  winner_games  loser_games  percentage_of_winners  \
0    01           16             0           16                0.00000   
1    02           16             0           16                0.00000   
2    03           16             0           16                0.00000   
3    04           16             0           16                0.00000   
4    05           14             0           14                0.00000   
5    06           15             0           15                0.00000   
6    07           13             0           13                0.00000   
7    08           16             0           16                0.00000   
8    09           14             0           14                0.00000   
9    10           14             0           14                0.00000   
10   11           14             0           14                0.00000   
11   12           16             0           16                0.00000   
12   13           13             0    

In [139]:
# 6
query = '''--SELECT * FROM betting_log 
--JOIN Schedule on betting_log.game_code = schedule.game_code limit 5

WITH CTE AS
  (SELECT home_team_id,
    COUNT(CASE WHEN score_home > score_away THEN 1 END) AS home_wins, 
    COUNT(home_team_id) as total_home_games,
    COUNT(CASE WHEN score_home < score_away THEN 1 END) AS home_lost,
    COUNT(CASE WHEN winner_line = 'home' THEN 1 END) AS home_beat_spread
  FROM Schedule 
  GROUP BY home_team_id),
CTE2 AS
  (SELECT away_team_id,
    COUNT(CASE WHEN score_away > score_home THEN 1 END) AS away_wins,
    COUNT(away_team_id) as total_away_games,
    COUNT(CASE WHEN score_away < score_home THEN 1 END) AS away_lost,
    COUNT(CASE WHEN winner_line = 'away' THEN 1 END) AS away_beat_spread
  FROM Schedule 
  GROUP BY away_team_id),
CTE3 AS 
  (SELECT t.team_id, t.team_name,
  COUNT(CASE WHEN b.bet_on = t.team_name THEN 1 END) as total_bets_for,
  COUNT(CASE WHEN b.bet_on <> t.team_name AND b.bet_on NOT IN ('over', 'under', 'push') THEN 1 END) as total_bets_against
  FROM teams t
    JOIN schedule s on (t.team_id = s.home_team_id OR t.team_id = s.away_team_id)
    JOIN betting_log b on s.game_code = b.game_code
  WHERE b.bet_on NOT IN ('over', 'under', 'push') 
  GROUP BY t.team_id, t.team_name)
SELECT t.team_name,
  c.total_home_games + c2.total_away_games as total_games, 
  c.home_wins + c2.away_wins as total_wins, 
  c.home_lost + c2.away_lost as total_lost,
  c.home_beat_spread + c2.away_beat_spread as total_beat_spread,
  c3.total_bets_for, c3.total_bets_against
FROM teams t 
  JOIN CTE c ON t.team_id = c.home_team_id
  JOIN CTE2 c2 on t.team_id = c2.away_team_id
  JOIN CTE3 c3 on t.team_name = c3.team_name
ORDER BY t.team_name;'''

df = pd.read_sql(query, n_conn)
print(df)

                   team_name  total_games  total_wins  total_lost  \
0          Arizona Cardinals          150          64          84   
1            Atlanta Falcons          152          71          81   
2            Baltimore Colts          150          71          78   
3           Baltimore Ravens          154          90          64   
4            Boston Patriots          160         100          60   
5              Buffalo Bills          157          93          64   
6          Carolina Panthers          151          65          86   
7              Chicago Bears          149          58          91   
8         Cincinnati Bengals          154          73          79   
9           Cleveland Browns          150          55          94   
10            Dallas Cowboys          154          88          66   
11            Denver Broncos          150          67          83   
12             Detroit Lions          151          65          84   
13         Green Bay Packers      

In [111]:
# 7.a
import statsmodels.api as sm

# calc customers value
query = '''SELECT b.customer_id,
  SUM(CASE result
    WHEN 'win' THEN b.commission - (b.bet_amount * 2)
    WHEN 'loss' THEN b.commission + b.bet_amount
    ELSE b.commission - b.bet_amount
  END) AS net_value,
  c.age, c.customer_type, c.customer_income, c.household_size
FROM customer AS c JOIN betting_log AS b
  ON c.customer_id = b.customer_id
GROUP BY b.customer_id, c.age, c.customer_type, c.customer_income, c.household_size; '''

df = pd.read_sql(query, n_conn)

# dummy variables
# with multicollinearity 
df_dummies = pd.get_dummies(df, columns = ['customer_type'])
print(df_dummies.corr())
print('\n', "There is a multicollinearity issue between the customer_type columns of local, phone, and online because two columns always perfectly predict the other.", sep='')
print("The value between customer_type_local and customer_type_phone is -0.46, and the value between customer_type_local and customer_type_online is -0.77", '\n')

# without multicollinearity
df_dummies = pd.get_dummies(df, columns = ['customer_type'], drop_first=True, dtype=int)
print(df_dummies.corr())
print('\n', "There is no issue here because we added the 'drop_first=True' parameter. There are only two columns now instead of three (i.e. customer_type_local was dropped because it has the highest correlation to the other two customer_type columns). This is fine because we know if both columns have a 0 value then that indicates the customer type for that row is local.", '\n', sep='')

                      customer_id  net_value       age  customer_income  \
customer_id              1.000000  -0.000700 -0.010379         0.068920   
net_value               -0.000700   1.000000 -0.003548        -0.247747   
age                     -0.010379  -0.003548  1.000000        -0.003367   
customer_income          0.068920  -0.247747 -0.003367         1.000000   
household_size          -0.005875  -0.090860 -0.003167        -0.010054   
customer_type_local      0.002221   0.051839  0.003520         0.020711   
customer_type_online     0.000470  -0.097238  0.012831        -0.011735   
customer_type_phone     -0.004076   0.056486 -0.023397        -0.015428   

                      household_size  customer_type_local  \
customer_id                -0.005875             0.002221   
net_value                  -0.090860             0.051839   
age                        -0.003167             0.003520   
customer_income            -0.010054             0.020711   
household_size     

In [112]:
# 7.b
# first model

# independent variables
x = df_dummies[['age', 'customer_income', 'household_size', 'customer_type_online', 'customer_type_phone']]
# dependent variables 
y = df_dummies['net_value']

# add constant to IV's
x = sm.add_constant(x)

# add the OLS engine and .predict()
model = sm.OLS(y, x).fit()
predictions = model.predict(x)

# print model sum
print_model = model.summary()
print(print_model)

                            OLS Regression Results                            
Dep. Variable:              net_value   R-squared:                       0.081
Model:                            OLS   Adj. R-squared:                  0.079
Method:                 Least Squares   F-statistic:                     35.16
Date:                Sat, 06 Dec 2025   Prob (F-statistic):           1.46e-34
Time:                        13:59:35   Log-Likelihood:                -25944.
No. Observations:                2000   AIC:                         5.190e+04
Df Residuals:                    1994   BIC:                         5.193e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 4.137e+04 

In [126]:
# 7.c
query = ('''
    SELECT b.customer_id,
      SUM(CASE result
        WHEN 'win' THEN b.commission - (b.bet_amount * 2)
        WHEN 'loss' THEN b.commission + b.bet_amount
        ELSE b.commission - b.bet_amount
      END) AS net_value,
      c.age, c.customer_type, c.customer_income, c.household_size, c.mode_color, c.customer_since
    FROM customer AS c JOIN betting_log AS b
      ON c.customer_id = b.customer_id
    GROUP BY b.customer_id, c.age, c.customer_type, c.customer_income, c.household_size, c.mode_color, c.customer_since;
''')
df = pd.read_sql(query, n_conn)

# create mode_color categorical grouping BEFORE creating dummies
df['mode_color_category'] = 'other'  # Default for green, yellow
df.loc[df['mode_color'].isin(['purple', 'blue', 'black']), 'mode_color_category'] = 'negative'
df.loc[df['mode_color'].isin(['white']), 'mode_color_category'] = 'positive'

# create dummy variables for customer_type and mode_color_category
df_dummies = pd.get_dummies(df, columns=['customer_type', 'mode_color_category'], drop_first=False, dtype=int)

# household_size grouping 
df_dummies['household_size_below_5'] = (df_dummies['household_size'] <= 4).astype(int)

# drop multicol cols
df_dummies.drop('customer_type_local', axis=1, inplace=True)
df_dummies.drop('household_size', axis=1, inplace=True)
df_dummies.drop('mode_color_category_other', axis=1, inplace=True)
df_dummies.drop('customer_id', axis=1, inplace=True)


# Print correlation matrix - specify numeric cols only
print(df_dummies[['net_value', 'customer_income', 'customer_since', 'customer_type_online', 'customer_type_phone', 'mode_color_category_negative', 'mode_color_category_positive', 'household_size_below_5']].corr())

# label IV's
x = df_dummies[[
    'customer_income',
    'customer_since',
    'customer_type_online',
    'customer_type_phone',
    'mode_color_category_negative',  # This is your blue/black/purple group
    'mode_color_category_positive',  # This is your red/white/orange group
    'household_size_below_5'
]]

# label DV
y = df_dummies['net_value']

# Add constant
x = sm.add_constant(x)

# Fit model
model = sm.OLS(y, x).fit()

# Print output
print(model.summary())

                              net_value  customer_income  customer_since  \
net_value                      1.000000        -0.247747        0.117339   
customer_income               -0.247747         1.000000       -0.005408   
customer_since                 0.117339        -0.005408        1.000000   
customer_type_online          -0.097238        -0.011735        0.027774   
customer_type_phone            0.056486        -0.015428        0.048285   
mode_color_category_negative  -0.068070        -0.027985        0.041689   
mode_color_category_positive   0.072872        -0.011777       -0.004118   
household_size_below_5         0.119971         0.007672       -0.013157   

                              customer_type_online  customer_type_phone  \
net_value                                -0.097238             0.056486   
customer_income                          -0.011735            -0.015428   
customer_since                            0.027774             0.048285   
customer_type_o